<a href="https://colab.research.google.com/github/Mohd-Abdul-Muqeet/FlyRank-AI/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohd-Abdul-Muqeet/FlyRank-AI-Week1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
Ranking Signal Analysis lane, the method choice should be guided by the nature of your signals (CTR, impressions, position) and the goal: understanding which signals drive engagement and whether a model can beat the baseline.
Recommended Method: Logistic Regression + Permutation Importance
CTR is naturally a binary/ratio outcome (clicked vs. not clicked), making logistic regression a strong fit.

It provides interpretable coefficients — you can directly see how impressions, position, and other signals affect the probability of a click.

It’s lightweight, stable, and less prone to overfitting compared to complex models like Gradient Boosting, which is important when your dataset may have uneven distributions across queries or positions.

Permutation Importance?

Helps you quantify feature importance in a model-agnostic way.

Perfect for your lane because you want to know: Does position dominate CTR prediction? Do impressions add marginal value?

It ties directly to your signal audit work — showing which signals actually matter once modeled.



In [3]:
!pip install -U datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 21.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5


In [1]:
from huggingface_hub import login

login()

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")

from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")
ds

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client/query/session

Why: CTR and position are highly dependent on the query or client context. If you randomly split, the model could “cheat” by memorizing query-specific CTR patterns.

Grouped validation ensures that when you test, the model is seeing new queries/clients rather than just repeats of the same distribution. This makes your evaluation honest for ranking signals.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

ds= list(ds.take(10000))
ds = pd.DataFrame(ds)
print(ds.columns.tolist())
ds.head()

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df_filtered = ds[['gsc_impressions','gsc_clicks', 'gsc_avg_position', 'ga4_total_engagement_sec','sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai' ]]

df_filtered["CTR"] = df_filtered["gsc_clicks"] / df_filtered["gsc_impressions"]
df_filtered = df_filtered.sort_values(by="CTR", ascending=False)
df_filtered.head(20)

/tmp/ipykernel_1859/780685766.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["CTR"] = df_filtered["gsc_clicks"] / df_filtered["gsc_impressions"]


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,CTR
9410,1,1,0.0,0,0,0,0,0,0,0,1.0
1460,1,1,2.0,0,0,0,0,0,0,0,1.0
8779,1,1,2.0,0,0,0,0,0,0,0,1.0
8699,1,1,1.0,0,0,0,0,0,0,0,1.0
6192,1,1,44.0,0,0,0,0,0,0,0,1.0
6221,2,2,4.5,0,0,0,0,0,0,0,1.0
6874,1,1,4.0,0,0,0,0,0,0,0,1.0
6871,1,1,57.0,0,0,0,0,0,0,0,1.0
8979,1,1,11.0,0,0,0,0,0,0,0,1.0
6658,1,1,0.0,0,0,0,0,0,0,0,1.0


In [ ]:
df_filtered.isnull().sum()

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [11]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
Data leakage: The model probably had access to features that directly encode the target (e.g., gsc_clicks or CTR itself sneaking into X). If CTR is derived from clicks/impressions, and you left gsc_clicks in the feature set, the model can trivially reconstruct the target.

Trivial target: If you binarized CTR as CTR > 0, and most rows are either all 0 or all 1 with a perfectly correlated feature (like impressions or clicks), the model will score perfectly.

Split issue: If you used a random split instead of grouped-by-client, the same client/query patterns may appear in both train and test, so the model is just memorizing.

In [12]:
# Make predictions on test data
y_pred = model.predict(X_test)

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Accuracy:", accuracy_score(y_test, y_pred)*100,'%')
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 100.0 %
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.